# Estatísticas por Copa e por seleção

## Task 3: calcular as estatísticas por Copa e por seleção

## O que a transmissão precisa

1. **Por Copa:** partidas, gols, média de gols por partida, empates e público médio.
2. **Por seleção, em cada Copa:** jogos, vitórias, empates, derrotas, gols marcados, gols sofridos e saldo.

In [1]:
import sys
sys.path.insert(0, "../../src")

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("arena-estatisticas")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "../../data/warehouse")
    .getOrCreate()
)

RAW = "../../data/raw"

spark

A célula abaixo já vem pronta: é o mesmo código dos vídeos 2.1 (leitura e limpeza) e 2.2 (padronização), na mesma
ordem.

In [2]:
# vídeo 2.1: leitura e limpeza
from wc_pipeline.schemas import MATCHES

partidas_brutas = (
    spark.read
    .option("header", "true")
    .schema(MATCHES)
    .csv(f"{RAW}/WorldCupMatches.csv")
)

partidas_limpas = (
    partidas_brutas
    .filter(F.col("MatchID").isNotNull())
    .dropDuplicates(["MatchID"])
)

# vídeo 2.2: padronização
partidas = partidas_limpas.select(
    F.col("Year").alias("ano"),
    F.col("MatchID").alias("match_id"),
    F.col("Stage").alias("fase"),
    F.col("Home Team Name").alias("mandante"),
    F.col("Home Team Goals").alias("gols_mandante"),
    F.col("Away Team Goals").alias("gols_visitante"),
    F.col("Away Team Name").alias("visitante"),
    F.col("Attendance").alias("publico"),
    F.col("Datetime").alias("data_partida"),
)

partidas = partidas.withColumn(
    "publico",
    F.regexp_replace("publico", ",", "").cast("int"),
)

FORMATO_DATA = "dd MMM yyyy - HH:mm"

partidas = partidas.withColumn(
    "data_partida",
    F.to_timestamp("data_partida", FORMATO_DATA),
)

partidas = (
    partidas
    .withColumn("total_gols", F.col("gols_mandante") + F.col("gols_visitante"))
    .withColumn(
        "resultado",
        F.when(F.col("gols_mandante") > F.col("gols_visitante"), "mandante")
         .when(F.col("gols_mandante") < F.col("gols_visitante"), "visitante")
         .otherwise("empate"),
    )
)

partidas.printSchema()

root
 |-- ano: integer (nullable = true)
 |-- match_id: integer (nullable = true)
 |-- fase: string (nullable = true)
 |-- mandante: string (nullable = true)
 |-- gols_mandante: integer (nullable = true)
 |-- gols_visitante: integer (nullable = true)
 |-- visitante: string (nullable = true)
 |-- publico: integer (nullable = true)
 |-- data_partida: timestamp (nullable = true)
 |-- total_gols: integer (nullable = true)
 |-- resultado: string (nullable = false)



## Agregar sem agrupar: o histórico inteiro

O resumo de todas as Copas, a referência para comparar cada Copa com a média histórica. As funções de agregação
(`F.count`, `F.sum`, `F.avg`, `F.max`...) dentro de um `select` resumem a tabela numa linha.

In [3]:
partidas.select(
    F.count("*").alias("partidas"),
    F.sum("total_gols").alias("gols"),
    F.round(F.avg("total_gols"), 2).alias("media_gols"),
    F.round(F.avg("publico"), 0).alias("publico_medio"),
).show()

+--------+----+----------+-------------+
|partidas|gols|media_gols|publico_medio|
+--------+----+----------+-------------+
|     964|2720|      2.82|      44859.0|
+--------+----+----------+-------------+



`F.round(..., 2)` arredonda a média para duas casas.

## groupBy + agg: uma linha por Copa

`groupBy` separa os grupos; `agg` diz o que calcular em cada um.

In [4]:
#  partidas, gols, média de gols por partida, empates e público médio
por_copa = (
    partidas.groupBy("ano")
    .agg(
        F.count("*").alias("partidas"),
        F.sum("total_gols").alias("gols"),
        F.round(F.avg("total_gols"), 2).alias("media_gols"),
        F.sum(F.when(F.col("resultado") == "empate", 1).otherwise(0)).alias("empates"),
        F.round(F.avg("publico"), 0).alias("publico_medio"),
    )
    .orderBy("ano")
)
por_copa.show(20)

+----+--------+----+----------+-------+-------------+
| ano|partidas|gols|media_gols|empates|publico_medio|
+----+--------+----+----------+-------+-------------+
|1930|      18|  70|      3.89|      0|      32808.0|
|1934|      17|  70|      4.12|      1|      21353.0|
|1938|      18|  84|      4.67|      3|      20872.0|
|1950|      22|  88|       4.0|      3|      47511.0|
|1954|      26| 140|      5.38|      2|      29562.0|
|1958|      35| 126|       3.6|     10|      23423.0|
|1962|      32|  89|      2.78|      5|      27912.0|
|1966|      32|  89|      2.78|      5|      48848.0|
|1970|      32|  95|      2.97|      5|      50124.0|
|1974|      38|  97|      2.55|     10|      49099.0|
|1978|      38| 102|      2.68|      9|      40679.0|
|1982|      52| 146|      2.81|     17|      40572.0|
|1986|      52| 132|      2.54|     14|      46039.0|
|1990|      52| 115|      2.21|     12|      48389.0|
|1994|      52| 141|      2.71|     11|      68991.0|
|1998|      64| 171|      2.

`F.sum(F.when(...))` **conta com condição**: cada empate vale 1, o resto 0, e a soma de cada Copa é o número de
empates.

`avg`, `sum` e as outras agregações **ignoram nulos**; só o `count("*")` conta todas as linhas.

## Por seleção: o truque de virar a partida

Cada partida tem duas seleções, em colunas diferentes. Mudamos o **grão** (o que cada linha representa): cada partida
vira duas linhas, uma por seleção.

```
partidas:   mandante  gols_mandante  gols_visitante  visitante
             Brazil        3              1           Mexico

vira:      selecao   gols_pro   gols_contra
           Brazil       3           1
           Mexico       1           3
```

In [5]:
partidas.filter(F.col("mandante") == "England").show(2)
partidas.filter(F.col("visitante") == "England").show(2)

+----+--------+-------+--------+-------------+--------------+---------+-------+-------------------+----------+---------+
| ano|match_id|   fase|mandante|gols_mandante|gols_visitante|visitante|publico|       data_partida|total_gols|resultado|
+----+--------+-------+--------+-------------+--------------+---------+-------+-------------------+----------+---------+
|1950|      56|Group 2| England|            2|             0|    Chile|  29703|1950-06-25 15:00:00|         2| mandante|
|1954|      83|Group 4| England|            4|             4|  Belgium|  14000|1954-06-17 18:10:00|         8|   empate|
+----+--------+-------+--------+-------------+--------------+---------+-------+-------------------+----------+---------+
only showing top 2 rows
+----+--------+-------+-------------+-------------+--------------+---------+-------+-------------------+----------+---------+
| ano|match_id|   fase|     mandante|gols_mandante|gols_visitante|visitante|publico|       data_partida|total_gols|resultado

In [6]:
mandante = partidas.select(
    "ano",
    F.col("mandante").alias("selecao"),
    F.col("gols_mandante").alias("gols_pro"),
    F.col("gols_visitante").alias("gols_contra"),
)
visitante = partidas.select(
    "ano",
    F.col("visitante").alias("selecao"),
    F.col("gols_visitante").alias("gols_pro"),
    F.col("gols_mandante").alias("gols_contra"),
)

por_selecao = mandante.unionByName(visitante)
print("partidas:", partidas.count(), "| linhas por seleção:", por_selecao.count())

partidas: 964 | linhas por seleção: 1928


In [7]:
por_selecao.filter(F.col("selecao") == "England").show(4)

+----+-------+--------+-----------+
| ano|selecao|gols_pro|gols_contra|
+----+-------+--------+-----------+
|1950|England|       2|          0|
|1954|England|       4|          4|
|1954|England|       2|          0|
|1958|England|       2|          2|
+----+-------+--------+-----------+
only showing top 4 rows


`union` empilha **pela posição**; `unionByName`, **pelo nome**, que é o mais seguro.

In [8]:
# jogos, vitórias, empates, derrotas, gols marcados, gols sofridos e saldo, por Copa

por_selecao = por_selecao.withColumn(
    "resultado",
    F.when(F.col("gols_pro") > F.col("gols_contra"), "V")
     .when(F.col("gols_pro") < F.col("gols_contra"), "D")
     .otherwise("E"),
)

tabela_selecoes = (
    por_selecao.groupBy("ano", "selecao")
    .agg(
        F.count("*").alias("jogos"),
        F.sum(F.when(F.col("resultado") == "V", 1).otherwise(0)).alias("vitorias"),
        F.sum(F.when(F.col("resultado") == "E", 1).otherwise(0)).alias("empates"),
        F.sum(F.when(F.col("resultado") == "D", 1).otherwise(0)).alias("derrotas"),
        F.sum("gols_pro").alias("gols_pro"),
        F.sum("gols_contra").alias("gols_contra"),
    )
    .withColumn("saldo", F.col("gols_pro") - F.col("gols_contra"))
    .orderBy(F.desc("saldo"))
)
tabela_selecoes.show(5)

+----+-------+-----+--------+-------+--------+--------+-----------+-----+
| ano|selecao|jogos|vitorias|empates|derrotas|gols_pro|gols_contra|saldo|
+----+-------+-----+--------+-------+--------+--------+-----------+-----+
|1954|Hungary|    5|       4|      0|       1|      27|         10|   17|
|1950| Brazil|    6|       4|      1|       1|      22|          6|   16|
|2014|Germany|    7|       6|      1|       0|      18|          4|   14|
|2002| Brazil|    7|       7|      0|       0|      18|          4|   14|
|1998| France|    7|       6|      1|       0|      15|          2|   13|
+----+-------+-----+--------+-------+--------+--------+-----------+-----+
only showing top 5 rows


`groupBy("ano", "selecao")`: cada linha é uma seleção numa Copa. O `F.sum(F.when(...))` é o mesmo truque dos empates
por Copa: cada vitória (ou empate, ou derrota) vale 1, o resto 0. `F.desc` ordena do maior para o menor.

## Conferindo o resultado

O total de gols tem que ser o mesmo pelos dois caminhos: por Copa e por seleção. A ação `first()` traz a primeira
linha do resultado, e `[0]` pega o primeiro valor dela.

In [9]:
gols_por_copa = por_copa.select(F.sum("gols")).first()[0]
gols_por_selecao = tabela_selecoes.select(F.sum("gols_pro")).first()[0]

print("soma dos gols por Copa   :", gols_por_copa)
print("soma dos gols por seleção:", gols_por_selecao)

soma dos gols por Copa   : 2720
soma dos gols por seleção: 2720


## Para consultar depois

- **`groupBy("ano").count()`** é um atalho que só conta, com a coluna chamada `count`. O `agg` permite várias
  métricas, com os nomes que você escolher.
- **Outras agregações úteis:** `F.countDistinct("fase")` (quantos valores diferentes), `F.min`, `F.max` e
  `F.stddev`. `F.collect_set` e `F.collect_list` juntam os valores numa lista; cuidado em dado grande, porque uma chave
  com milhões de linhas vira uma lista gigante numa linha só.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [10]:
spark.stop()